## Step 1: Imports

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

## Step 2: Upload CSVs

Upload `cnn_embeddings_daily.csv` and `reef_minilm_embeddings_2018_2025.csv`

In [ ]:
uploaded = files.upload()

## Step 3: Load CSVs

In [ ]:
cnn_df = pd.read_csv("cnn_embeddings_daily.csv")
nlp_df = pd.read_csv("reef_minilm_embeddings_2018_2025.csv")

cnn_df["date"] = pd.to_datetime(cnn_df["date"])

print("CNN shape:", cnn_df.shape)
print("NLP shape:", nlp_df.shape)

## Step 4: Season helper

In [ ]:
def get_season(month):
    """Australian seasons."""
    if month in [12, 1, 2]:  return "Summer"
    elif month in [3, 4, 5]: return "Autumn"
    elif month in [6, 7, 8]: return "Winter"
    else:                     return "Spring"

def get_season_year(date):
    """
    Season year = the year the season STARTS in.
    Dec 2021 belongs to Summer 2021 (not 2022).
    """
    season = get_season(date.month)
    year   = date.year - 1 if (season == "Summer" and date.month == 12) else date.year
    return season, year

## Step 5: Average NLP embeddings across North + Central per month

In [ ]:
nlp_emb_cols = [c for c in nlp_df.columns if c.startswith("emb_")]

# Average North and Central embeddings for each month
nlp_avg = (
    nlp_df.groupby("period")[nlp_emb_cols]
    .mean()
    .reset_index()
)
nlp_avg.columns = ["period"] + [f"nlp_{c}" for c in nlp_emb_cols]

print("NLP averaged shape:", nlp_avg.shape)  # (96, 385)

## Step 6: Map each CNN day → season → average monthly NLP embeddings for that season

In [ ]:
# Add season info to CNN
cnn_df["season"], cnn_df["season_year"] = zip(*cnn_df["date"].apply(get_season_year))
cnn_df["month"] = cnn_df["date"].dt.month
cnn_df["year"]  = cnn_df["date"].dt.year

# Add period column to NLP for joining
nlp_avg["year"]  = nlp_avg["period"].str[:4].astype(int)
nlp_avg["month"] = nlp_avg["period"].str[5:].astype(int)
nlp_avg["season"] = nlp_avg["month"].apply(get_season)
nlp_avg["season_year"] = nlp_avg.apply(
    lambda r: r["year"] - 1 if (r["season"] == "Summer" and r["month"] == 12) else r["year"],
    axis=1
)

nlp_season_cols = [c for c in nlp_avg.columns if c.startswith("nlp_")]

# Average NLP embeddings across all months within each (season, season_year)
nlp_seasonal = (
    nlp_avg.groupby(["season", "season_year"])[nlp_season_cols]
    .mean()
    .reset_index()
)

print("Seasonal NLP shape:", nlp_seasonal.shape)  # (32, 386) = 4 seasons x 8 years
print(nlp_seasonal[["season","season_year"]].sort_values(["season_year","season"]).head(12).to_string())

## Step 7: Broadcast seasonal NLP onto every day and concatenate

In [ ]:
# Join seasonal NLP onto each CNN day via (season, season_year)
fused_df = cnn_df.merge(nlp_seasonal, on=["season", "season_year"], how="left")

# Check coverage
nlp_cols_in_fused = [c for c in fused_df.columns if c.startswith("nlp_")]
missing = fused_df[nlp_cols_in_fused[0]].isna().sum()
print(f"Days with NLP vector : {len(fused_df) - missing}")
print(f"Days missing NLP     : {missing}  (outside 2018-2025 NLP range)")

# Fill any missing NLP rows with zeros
fused_df[nlp_cols_in_fused] = fused_df[nlp_cols_in_fused].fillna(0.0)

# Final column order: date, split, label, cnn embeddings, nlp embeddings
cnn_emb_cols = [c for c in cnn_df.columns if c.startswith("emb_")]
final_cols   = ["date", "split", "label"] + cnn_emb_cols + nlp_cols_in_fused

fused_df = fused_df[final_cols].sort_values("date").reset_index(drop=True)

print(f"\nFused shape: {fused_df.shape}")
print(f"  CNN dims : {len(cnn_emb_cols)}")
print(f"  NLP dims : {len(nlp_cols_in_fused)}")
print(f"  Total dims: {len(cnn_emb_cols) + len(nlp_cols_in_fused)}")
fused_df.head(3)

## Step 8: Save and download

In [ ]:
OUTPUT_PATH = "/content/reef_fused_embeddings.csv"
fused_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(fused_df)} rows -> {OUTPUT_PATH}")
print(f"Columns: date, split, label + {len(cnn_emb_cols)} CNN dims + {len(nlp_cols_in_fused)} NLP dims")

from google.colab import files as colab_files
colab_files.download(OUTPUT_PATH)